# CNN PPO Quantization Benchmark

Trying various optimizations for our massive CNN PPO (quantization to various precisons, compilation). Results should be preserved. Results from running on Google T4 GPU (TPU?) on colab.

In [ ]:
import torch
import torch.nn as nn
import time, os

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


In [ ]:
#model defn lifted from CNN ppo def
import numpy as np

class ActorCritic(nn.Module):
    def __init__(self, n_actions=7):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        self.fc = nn.Sequential(nn.Linear(3136, 512), nn.ReLU())
        self.policy_head = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, n_actions)
        )
        self.value_head = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 1)
        )
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.policy_head[-1].weight, gain=0.01)
        nn.init.orthogonal_(self.value_head[-1].weight, gain=1.0)

    def forward(self, x):
        h = self.fc(self.encoder(x).flatten(1))
        return self.policy_head(h), self.value_head(h).squeeze(-1)

In [ ]:
#Load checkpoint
ckpt = torch.load('to_quant.pt', map_location='cpu')
weights = ckpt['model'] if 'model' in ckpt else ckpt

model_fp32 = ActorCritic()
model_fp32.load_state_dict(weights)
model_fp32.eval()
print(f'Loaded. Params: {sum(p.numel() for p in model_fp32.parameters()):,}')

Loaded. Params: 2,213,544


In [ ]:
#Benchmark
def benchmark(model, device, n=5000, dtype=torch.float32):
    model = model.to(device)
    x = torch.rand(1, 4, 84, 84, dtype=dtype, device=device)
    with torch.no_grad():
        for _ in range(200):   # warmup
            model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n):
            model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return n / elapsed, elapsed / n * 1e6

def saved_mb(model):
    torch.save(model.state_dict(), '/tmp/_tmp.pt')
    return os.path.getsize('/tmp/_tmp.pt') / 1e6

def row(label, fps, us, mb, fps_base):
    print(f'  {label:<30s}  {fps:>8,.0f} inf/s  {us:>7.1f} µs  {mb:>6.2f} MB  [{fps/fps_base:.2f}x]')

cpu = torch.device('cpu')
gpu = torch.device('cuda') if torch.cuda.is_available() else None

print(f'{"":30s}  {"inf/s":>8}  {"µs/inf":>7}  {"MB":>6}  speedup')
print('─' * 68)

fps_base, us_base = benchmark(model_fp32, cpu)
mb_base = saved_mb(model_fp32)
row('float32  CPU (baseline)', fps_base, us_base, mb_base, fps_base)

if gpu:
    fps_gpu, us_gpu = benchmark(model_fp32, gpu)
    row('float32  GPU', fps_gpu, us_gpu, mb_base, fps_base)

                                   inf/s   µs/inf      MB  speedup
────────────────────────────────────────────────────────────────────
  float32  CPU (baseline)              832 inf/s   1201.3 µs    8.86 MB  [1.00x]
  float32  GPU                       1,633 inf/s    612.2 µs    8.86 MB  [1.96x]


In [ ]:
#Float16 on GPU (T4 for experiments)
if gpu:
    model_fp16 = ActorCritic()
    model_fp16.load_state_dict(weights)
    model_fp16.half().eval()
    fps_fp16, us_fp16 = benchmark(model_fp16, gpu, dtype=torch.float16)
    mb_fp16 = saved_mb(model_fp16)
    row('float16  GPU', fps_fp16, us_fp16, mb_fp16, fps_base)
else:
    print('No GPU — skipping float16 GPU benchmark')

  float16  GPU                       1,499 inf/s    667.1 µs    4.43 MB  [1.74x]


In [ ]:
#Try torch compiling. Also useless
if gpu and hasattr(torch, 'compile'):
    model_compiled = ActorCritic()
    model_compiled.load_state_dict(weights)
    model_compiled.eval().to(gpu)
    model_compiled = torch.compile(model_compiled)
    fps_c, us_c = benchmark(model_compiled, gpu)
    row('float32  GPU + torch.compile', fps_c, us_c, mb_base, fps_base)
else:
    print('Skipping torch.compile')

W0429 04:46:35.197000 2047 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


  float32  GPU + torch.compile       1,591 inf/s    628.4 µs    8.86 MB  [1.85x]
